# Aula 09 | Avaliar RAG: recuperar e responder

**Inteligência Artificial para Decisão** · Marketing e Finanças · notebook local.

**Pergunta:** Como distinguir uma busca fraca de uma resposta mal fundamentada?

**Procedimento:** execute as células, confira os resultados e modifique uma hipótese. Não há chamadas a serviços externos. Todos os dados são sintéticos ou fictícios.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

# Localiza a raiz do curso, inclusive ao abrir o notebook a partir da pasta da aula.
def achar_raiz():
    origem = Path.cwd().resolve()
    for pasta in (origem, *origem.parents):
        if (pasta / "dados" / "README.md").exists() and (pasta / "src").exists():
            return pasta
    raise FileNotFoundError("Abra o notebook dentro do repositório extraído, preservando dados/ e src/.")

ROOT = achar_raiz()
sys.path.insert(0, str(ROOT / "src"))
print("Repositório:", ROOT)


## Avaliar retrieval, não geração

In [ ]:
from iad.metrics import recall_at_k, mean_reciprocal_rank
from iad.retrieval import load_corpus,chunk_headings,TfidfIndex
base=load_corpus(ROOT/'dados/relatorios');base.update(load_corpus(ROOT/'dados/faq'));base.update(load_corpus(ROOT/'dados/comunicados'))
idx=TfidfIndex(chunk_headings(base,max_chars=1250))
questoes=pd.read_csv(ROOT/'dados/perguntas_rag.csv')
rankings=[[h.doc_id for h in idx.search(q,k=20)] for q in questoes.pergunta]
gold=[None if x=='nenhuma' else x for x in questoes.documento_esperado]
print('Recall@1:',recall_at_k(rankings,gold,1))
print('Recall@3:',recall_at_k(rankings,gold,3))
print('MRR:',mean_reciprocal_rank(rankings,gold))

## Diagnóstico por caso

In [ ]:
for i,row in questoes.iterrows():
    print('Q',i+1, 'gabarito:',row.documento_esperado, 'top2:',rankings[i][:2])
print('Para questões sem fonte, a avaliação da resposta exige verificar abstenção.')

## Síntese individual

1. Qual pergunta este experimento responde e qual **não** responde?
2. Que valores foram calculados e quais foram apenas interpretados?
3. Que alteração você realizou? Registre a saída e explique sua escolha.
4. Qual erro seria especialmente relevante para uma decisão em **Marketing e Finanças**?

**Entrega:** Relatório de avaliação com tabela pergunta-fonte-evidência-resposta.